Importação das bibliotecas

In [1]:
from pathlib import Path
import urllib.request
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

Carregar e usar os dados

In [5]:
class CarregadorDados:

    def __init__(self, url, arquivos, pasta="data"):
        self.url = url
        self.arquivos = arquivos
        self.pasta = Path(pasta)
        self.pasta_raw = self.pasta / "raw"
        self.pasta_processed = self.pasta / "processed"
        self.pasta_raw.mkdir(parents=True, exist_ok=True)
        self.pasta_processed.mkdir(parents=True, exist_ok=True)

    def baixar(self):
        zip_path = self.pasta / "occupancy_data.zip"

        try:
            urllib.request.urlretrieve(self.url, zip_path)
        except Exception as erro:
            raise RuntimeError(
                "Não foi possível baixar a base. Baixe o zip na página da UCI "
                "e extraia os arquivos .txt em data/raw."
            ) from erro

        with zipfile.ZipFile(zip_path) as z:
            z.extractall(self.pasta_raw)

        zip_path.unlink()

    def carregar(self):
        faltando = [a for a in self.arquivos if not (self.pasta_raw / a).exists()]

        if faltando:
            print("Baixando:", faltando)
            self.baixar()

        return {
            Path(a).stem: pd.read_csv(self.pasta_raw / a).reset_index(drop=True)
            for a in self.arquivos
        }


carregador = CarregadorDados(
    url="https://archive.ics.uci.edu/ml/machine-learning-databases/00357/occupancy_data.zip",
    arquivos=["datatraining.txt", "datatest.txt", "datatest2.txt"],
    pasta="../data"
)

dfs = carregador.carregar()
df = dfs["datatest"]

print("Base de dados carregada com sucesso!")
df.head()

Base de dados carregada com sucesso!


,date,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,2015-02-02 14:19:00,23.7000,26.272,585.200000,749.200000,0.004764,1
1,2015-02-02 14:19:59,23.7180,26.290,578.400000,760.400000,0.004773,1
2,2015-02-02 14:21:00,23.7300,26.230,572.666667,769.666667,0.004765,1
3,2015-02-02 14:22:00,23.7225,26.125,493.750000,774.750000,0.004744,1
4,2015-02-02 14:23:00,23.7540,26.200,488.600000,779.000000,0.004767,1


# Fase 1: Análise Exploratória (EDA)

Nesta fase olho para a estrutura da base, o resumo estatístico e o comportamento das variáveis, para decidir como tratar os dados e modelar nas próximas fases.

## Dimensões, tipos e resumo estatístico

In [6]:
print("Linhas e colunas:", df.shape)
print()
df.info()
print()
print(df.describe())

Linhas e colunas: (2665, 7)

<class 'pandas.DataFrame'>
RangeIndex: 2665 entries, 0 to 2664
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           2665 non-null   str    
 1   Temperature    2665 non-null   float64
 2   Humidity       2665 non-null   float64
 3   Light          2665 non-null   float64
 4   CO2            2665 non-null   float64
 5   HumidityRatio  2665 non-null   float64
 6   Occupancy      2665 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 145.9 KB

       Temperature     Humidity        Light          CO2  HumidityRatio  \
count  2665.000000  2665.000000  2665.000000  2665.000000    2665.000000   
mean     21.433876    25.353937   193.227556   717.906470       0.004027   
std       1.028024     2.436842   250.210906   292.681718       0.000611   
min      20.200000    22.100000     0.000000   427.500000       0.003303   
25%      20.650000    23.260000     0.0

## Coluna date

A base tem uma coluna com a data e a hora de cada leitura. Antes de decidir o que fazer com ela, confiro o formato e o período coberto.

In [7]:
df["date"] = pd.to_datetime(df["date"])

print(df["date"].min(), "até", df["date"].max())
print(df["date"].diff().value_counts().head())

2015-02-02 14:19:00 até 2015-02-04 10:43:00
date
0 days 00:01:00    1596
0 days 00:00:59     534
0 days 00:01:01     534
Name: count, dtype: int64


A coluna estava como texto e foi convertida para datetime.
As leituras são feitas em intervalos regulares de cerca de um minuto, em um período de poucos dias.
Vou descartá-la porque o KNN e a árvore de decisão serão treinados só com as variáveis numéricas dos sensores, e a data indica o momento da leitura, não o comportamento do ambiente.

In [8]:
df = df.drop(columns="date")
df.head()

,Temperature,Humidity,Light,CO2,HumidityRatio,Occupancy
0,23.7000,26.272,585.200000,749.200000,0.004764,1
1,23.7180,26.290,578.400000,760.400000,0.004773,1
2,23.7300,26.230,572.666667,769.666667,0.004765,1
3,23.7225,26.125,493.750000,774.750000,0.004744,1
4,23.7540,26.200,488.600000,779.000000,0.004767,1
